# Cell segmentation refinement — controlled ablations

This notebook tests **spectrally corrupted ground-truth segmentations**, not U-Net predictions. Every energy configuration starts from **identical precomputed corrupted probability images**, then uses the same plain Euclidean seeded expansion with a **25-pixel cap**. We plot pooled instance F1 versus refinement iteration.

**Six conditions:** no refinement, symmetric MM, asymmetric MM (a_uz=3, a_uw=2, a_wz=1), templates only, symmetric MM + templates, asymmetric MM + templates.

**Required for the template conditions:** the original eight 21×21 painted PNGs. They were not in the latest attachments. MM-only conditions run without them. A saved template `best.pt` is optional; without one, template strengths are held fixed at 0.05. The notebook does not train new weights.


## 1. Set up project files

**Colab:** Upload the supplied `segmentation_refinement_tests.zip`, unzip it into `/content`, and run this notebook. You can also keep the unzipped folder on Drive; change `PROJECT_DIR` below. **Local Jupyter:** set `PROJECT_DIR` to the folder containing the Python files.


In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.util

PROJECT_DIR = Path('/content/segmentation_refinement_tests') if Path('/content').is_dir() else Path.cwd()
ARCHIVE = Path('/content/segmentation_refinement_tests.zip')
if not (PROJECT_DIR / 'run_refinement_study.py').exists() and ARCHIVE.exists():
    import zipfile
    with zipfile.ZipFile(ARCHIVE) as z:
        z.extractall('/content')
if not (PROJECT_DIR / 'run_refinement_study.py').exists():
    raise FileNotFoundError(f'Could not find run_refinement_study.py under {PROJECT_DIR}. Upload/unzip the package or change PROJECT_DIR.')
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('Working directory:', Path.cwd())


## 2. Dependencies and smoke tests

The synthetic fluorescence generator uses `psfmodels`, `numba`, and `scikit-image`. Installing may be unnecessary if these packages already exist in your environment.

In [ ]:
packages = {'torch':'torch', 'numpy':'numpy', 'scipy':'scipy', 'matplotlib':'matplotlib',
            'PIL':'pillow', 'skimage':'scikit-image', 'numba':'numba',
            'psfmodels':'psfmodels', 'pytest':'pytest'}
missing = [pip for module,pip in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing:', missing)
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
else:
    print('Dependencies already installed')
subprocess.run([sys.executable,'-m','pytest','-q','test_refinement_study.py'], check=True)


## 3. Templates and optional strengths checkpoint

If you have the original eight painted-template PNGs, place them in `PROJECT_DIR/templates`. Optionally specify a checkpoint saved by the previous `train_template_energy.py`; it will be loaded with a strict name/order check. A checkpoint trained under symmetric MM is **not retrained** here for asymmetric MM. For a first pilot, fixed 0.05 strengths are easiest to interpret.

In [ ]:
from refinement_ablations import ABLATIONS
TEMPLATE_DIR = PROJECT_DIR / 'templates'
TEMPLATE_CHECKPOINT = ''  # e.g. '/content/drive/MyDrive/project/best.pt'
found = sorted(p.name for p in TEMPLATE_DIR.glob('*.png')) if TEMPLATE_DIR.exists() else []
print('Template files:', found)
if not found:
    print('No template PNGs found. The notebook will run the three template-free controls.')
    SELECTED = ABLATIONS[:3]
else:
    SELECTED = ABLATIONS
print('Ablations:', [a.name for a in SELECTED])


## 4. Choose a pilot or full validation run

Pilot defaults here are modest for cost. A proper multi-noise presentation run could use `n_volumes=8`, `n_slices_per_volume=8`, `horizons=(0,25,50,100,200,300,400)` and possibly noise `(0.4,0.8,1.2,1.6)` on an accelerator. The runtime can be substantial because the refinement step differentiates through the energy at each iteration. Use a NEW output directory for each run.

For a presentation, reserve separate volume seeds for final testing after choosing any hyperparameters. All ablations should share the same fixed examples.

In [ ]:
import torch
from run_refinement_study import StudyConfig, build_validation_data, run_study, plot_study

PILOT = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
settings = dict(
    output_dir=str(PROJECT_DIR / ('results_pilot' if PILOT else 'results_full')),
    device=DEVICE,
    domain='a', volume_seed=320000, corruption_seed=840000,
    n_volumes=1 if PILOT else 8,
    n_slices_per_volume=2 if PILOT else 8,
    image_size=192, z_slices=16, z_ratio=2., n_seeds=45,
    class_probs=(0.4,0.2,0.4),
    noise_levels=(0.8,) if PILOT else (0.4,0.8,1.2,1.6),
    blur_sigma=0.75,
    horizons=(0,10,25) if PILOT else (0,25,50,100,200,300,400),
    anchor_mu=0.4, step_size=0.02,
    lam=0.2, epsilon=0.5, cubic_weight=1.,
    watershed_radius=25., watershed_cost_weight=0.0,  # cost weight unused by Euclidean expansion
    template_dir=str(TEMPLATE_DIR),template_checkpoint=TEMPLATE_CHECKPOINT,
    orientations=16, fixed_strength=0.05, strength_cap=0.3,
    save_previews=True,
)
config = StudyConfig(**settings)
print('Device:', config.device, '| output:', config.output_dir)


## 5. Generate corrupted-GT examples **once**

The original images supply the **nonnuclear fluorescence cost field**; corrupted GT supplies only the initial class probabilities. Samples and noise seeds are fixed independently of the ablation.

In [ ]:
examples = build_validation_data(config)
print(f'{len(examples)} paired examples')

import matplotlib.pyplot as plt
ex = examples[0]
fig,axs=plt.subplots(1,4,figsize=(14,3.5),constrained_layout=True)
axs[0].imshow(ex['target'].numpy()==1,cmap='gray'); axs[0].set_title('GT nuclear mask')
axs[1].imshow(ex['original'].argmax(0),cmap='tab10',vmin=0,vmax=9); axs[1].set_title('Corrupted GT argmax')
axs[2].imshow(ex['original'][1],cmap='magma',vmin=0,vmax=1); axs[2].set_title('Corrupted nuclear prob.')
axs[3].imshow(ex['nonnuclear'],cmap='magma'); axs[3].set_title('Original nonnuclear image')
for ax in axs: ax.axis('off')
plt.show()


## 6. Run paired energy ablations and radius-limited seeded-expansion evaluation

Each case records F1/PQ, nuclear Dice before and after growing, merges/splits, and energy components at each horizon. The **no-refinement** case remains unchanged across iterations. The original GT nuclear mask is not expanded.

In [ ]:
rows, summary, traces = run_study(examples, config, selected=SELECTED)
print(f'Wrote {len(rows)} per-slice metric rows and {len(summary)} summary rows')


## 7. Plot F1 versus refinement iteration

In [ ]:
figures=plot_study(summary,config.output_dir)
for fig in figures:
    display(fig)
    plt.close(fig)


## 8. Compare final-horizon F1 improvement over the same starting point

In [ ]:
import pandas as pd
s=pd.DataFrame(summary)
final_step=max(config.horizons)
last=s.loc[s.iteration==final_step,
    ['ablation','noise','n_slices','micro_instance_f1','delta_f1','mean_pq',
     'nuclear_dice_before_growth','nuclear_dice_after_growth','merges','splits']]
display(last.sort_values(['noise','micro_instance_f1'],ascending=[True,False]))


## 9. Energy-term diagnostics

In [ ]:
t=pd.DataFrame(traces)
columns=[x for x in ['dirichlet','triple_well','templates','anchor','total'] if x in t]
display(t.groupby(['ablation','noise','iteration'],as_index=False)[columns].mean().tail(30))


## 10. View saved previews and export results

`slice_metrics.csv`, `summary.csv`, `energy_traces.csv`, `config.json`, example previews, and `f1_vs_steps_noise_*.png` are in `config.output_dir`. You can archive the complete output folder in Drive or download a zip in Colab.

In [ ]:
from IPython.display import display, Image as DisplayImage
from pathlib import Path
previews=sorted(Path(config.output_dir).glob('*preview.png'))
print('Saved preview images:', len(previews))
for p in previews[:4]:
    print(p.name)
    display(DisplayImage(filename=str(p),width=850))
import shutil
archive=shutil.make_archive(str(Path(config.output_dir)), 'zip', config.output_dir)
print('Results archive:',archive)


## Interpretation notes

- **Pooled F1** is computed from summed TP, FP and FN over all validation slices for a given ablation and noise level. Per-slice scores are in the CSV.
- The seeded expansion is shared across conditions, but may itself be a powerful postprocessor. Always compare **t=0 watershed** against the same ablation after refinement.
- The asymmetric MM coefficients affect interfacial penalties, not a guaranteed nonnuclear halo; watch for nucleus shrinkage.
- The study is not evidence about generalization to a U-Net until you repeat it on actual model outputs.
- For robust uncertainty estimates, collect more **independent volumes** and bootstrap by volume rather than treating slices as independent samples.
